# سامانه پرسش و پاسخ تصویری به زبان فارسی (Persian Visual Question Answering)
## استقرار هسته هوش مصنوعی چندوجهی (Qwen2-VL) در Google Colab

این نوت‌بوک هسته پردازش هوش مصنوعی پروژه Persian-VQA را بر اساس نیازمندی‌های پروپوزال پیاده‌سازی می‌کند:
1. **بارگذاری مدل پیش‌آموزش‌دیده:** `Qwen/Qwen2-VL-7B-Instruct` (یا نسخه سبک‌تر `Qwen2-VL-2B-Instruct`)
2. **بهینه‌سازی حافظه VRAM برای Tesla T4:** کوانتیزاسیون ۴-بیتی NF4 برای وزن‌های زبانی و FP16 برای رمزگذار بینایی
3. **پیش‌پردازش و نرمال‌سازی تصویر و متن:** سقف استاندارد ۱۰۲۴×۹۸۰ پیکسل و پاک‌سازی نویسه‌های فارسی
4. **خروجی ساختاریافته JSON:** فیلدهای `answer`، `confidence` و `evidence` با پایبندی به شواهد دیداری
5. **درگاه ارتباطی FastAPI و تونل ngrok:** ارائه API اختصاصی جهت اتصال به بک‌اند سامانه

**راهنما:** از منوی **Runtime > Change runtime type** پردازنده **T4 GPU** را فعال کرده و سلول‌ها را به‌ترتیب اجرا کنید.

In [ ]:
# گام ۱: نصب بسته‌های مورد نیاز
!pip install -q -U "transformers>=4.45.0" "accelerate>=0.26.0" bitsandbytes torchvision pillow fastapi uvicorn pyngrok nest-asyncio pydantic
print("Dependencies installed successfully!")

In [ ]:
# گام ۲: وارد کردن کتابخانه‌ها و تنظیمات پایه
import io
import os
import gc
import re
import json
import base64
import time
import unicodedata
import threading
import nest_asyncio
from typing import Optional, Dict, Any

import torch
from PIL import Image
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from pyngrok import ngrok
import uvicorn

from transformers import (
    Qwen2VLForConditionalGeneration,
    Qwen2VLProcessor,
    BitsAndBytesConfig
)

# اعمال nest_asyncio برای اجرای Uvicorn در محیط نوت‌بوک
nest_asyncio.apply()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# گام ۳: بارگذاری مدل Qwen2-VL با کوانتیزاسیون ۴-بیتی NF4
# بر اساس پروپوزال، در صورت نیاز به استفاده در رم محدودتر می‌توان از Qwen/Qwen2-VL-2B-Instruct نیز استفاده کرد.
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
# جایگزین سبک‌تر در صورت کمبود حافظه:
# MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print(f"Loading model: {MODEL_ID} ...")

# تنظیمات کوانتیزاسیون ۴-بیتی NF4 طبق بند هـ و د بخش ۵ پروپوزال
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# بارگذاری پردازشگر با سقف استاندارد تصویر (حداکثر ۱۰۲۴×۹۸۰ پیکسل)
MAX_PIXELS = 1024 * 980
MIN_PIXELS = 256 * 256

processor = Qwen2VLProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS
)

try:
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
except Exception as e:
    print(f"Failed loading 7B model ({e}), falling back to 2B-Instruct...")
    MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
    processor = Qwen2VLProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )

model.eval()
print(f"Model {MODEL_ID} loaded successfully!")

In [ ]:
# گام ۴: نرمال‌سازی متن فارسی و قالب مهندسی پرامپت
ARABIC_TO_PERSIAN = {
    '\u064a': '\u06cc', '\u0649': '\u06cc', '\u0643': '\u06a9',
    '\u0629': '\u0647', '\u06c0': '\u0647', '\u0624': '\u0648', '\u0626': '\u06cc'
}
DIGITS_MAP = {
    '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4',
    '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9'
}

def normalize_fa(text: str) -> str:
    if not text: return ""
    text = unicodedata.normalize('NFKC', text)
    for a, p in ARABIC_TO_PERSIAN.items():
        text = text.replace(a, p)
    for d, s in DIGITS_MAP.items():
        text = text.replace(d, s)
    text = re.sub(r'[\u200c\u200b\u200d]+', '\u200c', text)
    return text.strip()

SYSTEM_PROMPT = """شما دستیار تخصصی پرسش و پاسخ تصویری به زبان فارسی هستید.
صرفاً بر اساس عناصر و شواهد قابل مشاهده در تصویر پاسخ دهید. از حدس زدن اجتناب نمایید.
اگر پاسخ قابل تشخیص نیست، دقیقاً بنویسید «قابل تشخیص نیست».
پاسخ باید حتماً به صورت یک شیء JSON با ساختار زیر بازگردانده شود:
{
  "answer": "پاسخ مستقیم و روان به فارسی",
  "confidence": "مطمئن | نسبتاً مطمئن | نامطمئن",
  "evidence": "اشاره کوتاه به بخشی از تصویر که پاسخ بر اساس آن استنتاج شد"
}"""

In [ ]:
# گام ۵: تابع استنتاج با مکانیزم محافظت از کمبود حافظه (OOM Protection)
generation_lock = threading.Lock()

def run_vqa_inference(image: Image.Image, question: str, level: int = 1, retry_count: int = 0) -> Dict[str, Any]:
    norm_q = normalize_fa(question)
    prompt_text = f"{norm_q}\nلطفاً پاسخ را دقیقاً در قالب JSON مشخص‌شده خروجی دهید."
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]
    
    with generation_lock:
        try:
            text_prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = processor(
                text=[text_prompt],
                images=[image],
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(model.device)
            
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.eos_token_id
                )
            
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
            
            # تمیزکاری کش GPU
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            # استخراج JSON از پاسخ مدل
            json_match = re.search(r'\{.*\}', output_text, re.DOTALL)
            if json_match:
                try:
                    parsed = json.loads(json_match.group(0))
                    return {
                        "answer": parsed.get("answer", output_text),
                        "confidence": parsed.get("confidence", "مطمئن"),
                        "evidence": parsed.get("evidence", "بررسی بصری محتوای تصویر"),
                        "level": level
                    }
                except json.JSONDecodeError:
                    pass
            
            return {
                "answer": output_text.strip(),
                "confidence": "نسبتاً مطمئن",
                "evidence": "تحلیل کلی کادر تصویر توسط مدل",
                "level": level
            }
            
        except torch.cuda.OutOfMemoryError:
            # مکانیزم محافظت OOM طبق بند ۵-الف پروپوزال: کاهش ۲۵٪ ابعاد تصویر و تکرار
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
            if retry_count < 1:
                print("CUDA OOM encountered. Reducing image resolution by 25% and retrying...")
                new_w = int(image.width * 0.75)
                new_h = int(image.height * 0.75)
                resized_img = image.resize((new_w, new_h), Image.Resampling.LANCZOS)
                return run_vqa_inference(resized_img, question, level, retry_count=retry_count + 1)
            else:
                return {
                    "answer": "حافظه گرافیکی برای پردازش تصویر با این وضوح کافی نبود.",
                    "confidence": "نامطمئن",
                    "evidence": "خطای محدودیت VRAM در شبیه‌ساز کولب",
                    "level": level
                }

In [ ]:
# ??? ?: ????? ???? ????? ngrok ? ????? ???? Uvicorn
# ???? ngrok ??? ?? ?? https://dashboard.ngrok.com/get-started/your-authtoken ??????.
NGROK_TOKEN = ""  # <-- ???? ??? ?? ????? ???? ???? ?? ?? Colab Secrets ????? ??????
if os.environ.get("NGROK_AUTHTOKEN"):
    NGROK_TOKEN = os.environ["NGROK_AUTHTOKEN"]

if NGROK_TOKEN:
    ngrok.kill()  # ???? ???????? ???? ?? ???? ????? ????
    ngrok.set_auth_token(NGROK_TOKEN)
    public_tunnel = ngrok.connect(8000)
    print("\n=======================================================")
    print(f"Public AI_SERVICE_URL: {public_tunnel.public_url}")
    print("??? ???? ?? ?? ????? ????? AI_SERVICE_URL ?????? ???? ????.")
    print("=======================================================\n")
else:
    print("???? ngrok ???? ???? ???. ???? ??? ???? ???? 8000 ???? ??????.")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()


In [ ]:
# گام ۷: ایجاد تونل عمومی ngrok و اجرای سرور Uvicorn
# توکن ngrok خود را از https://dashboard.ngrok.com/get-started/your-authtoken بگیرید.
NGROK_TOKEN = ""  # <-- توکن خود را اینجا قرار دهید یا در Colab Secrets تنظیم نمایید
if os.environ.get("NGROK_AUTHTOKEN"):
    NGROK_TOKEN = os.environ["NGROK_AUTHTOKEN"]

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_tunnel = ngrok.connect(8000)
    print("\n=======================================================")
    print(f"Public AI_SERVICE_URL: {public_tunnel.public_url}")
    print("این آدرس را در متغیر محیطی AI_SERVICE_URL بک‌اند قرار دهید.")
    print("=======================================================\n")
else:
    print("توکن ngrok وارد نشده است. سرور روی پورت محلی 8000 اجرا می‌شود.")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server.run()

In [ ]:
# گام ۸ (اختیاری): تست تعاملی مستقیم داخل کولب
# می‌توانید بدون اتصال به بک‌اند، یک تصویر را مستقیماً تست کنید:
from google.colab import files
print("یک تصویر تستی انتخاب کنید:")
uploaded = files.upload()
for filename in uploaded.keys():
    test_img = Image.open(filename).convert("RGB")
    test_q = "چه چیزهایی در این تصویر دیده می‌شود؟"
    test_res = run_vqa_inference(test_img, test_q, level=1)
    print("\n--- نتیجه استنتاج مدل ---")
    print(json.dumps(test_res, ensure_ascii=False, indent=2))